In [1]:
""" Linear Regression """

import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

In [2]:
# Keeping imports near  the top of a notebook makes the code more readable.

import statsmodels.api as sm

In [3]:
# Importing whole modules is too much,
# I can import only a few items from a given module. help the namespace clean.

from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF
from statsmodels.stats.anova import anova_lm

In [6]:
# Use line break \ to ease reability.

from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)

In [7]:
""" Inspecting Objects and Namespaces """

# dir() = a list of objects in a namespace

dir()

['In',
 'MS',
 'Out',
 'VIF',
 '_',
 '_1',
 '_2',
 '_3',
 '_6',
 '__',
 '__DW_SCOPE__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '__vsc_ipynb_file__',
 '_dh',
 '_i',
 '_i1',
 '_i2',
 '_i3',
 '_i4',
 '_i5',
 '_i6',
 '_i7',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'anova_lm',
 'exit',
 'get_ipython',
 'load_data',
 'np',
 'open',
 'pd',
 'poly',
 'quit',
 'sm',
 'subplots',
 'summarize']

In [8]:
A = np.array([3, 5, 11])
dir(A)

['T',
 '__abs__',
 '__add__',
 '__and__',
 '__array__',
 '__array_finalize__',
 '__array_function__',
 '__array_interface__',
 '__array_namespace__',
 '__array_priority__',
 '__array_struct__',
 '__array_ufunc__',
 '__array_wrap__',
 '__bool__',
 '__class__',
 '__class_getitem__',
 '__complex__',
 '__contains__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__delitem__',
 '__dir__',
 '__divmod__',
 '__dlpack__',
 '__dlpack_device__',
 '__doc__',
 '__eq__',
 '__float__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__iand__',
 '__ifloordiv__',
 '__ilshift__',
 '__imatmul__',
 '__imod__',
 '__imul__',
 '__index__',
 '__init__',
 '__init_subclass__',
 '__int__',
 '__invert__',
 '__ior__',
 '__ipow__',
 '__irshift__',
 '__isub__',
 '__iter__',
 '__itruediv__',
 '__ixor__',
 '__le__',
 '__len__',
 '__lshift__',
 '__lt__',
 '__matmul__',
 '__mod__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '_

In [9]:
# It show the method that can be used

A.sum()

np.int64(19)

In [10]:
""" Simple Linear Regression """

# Design matrices using the ModelSpec() transform from ISLP.models
# Boston housing datasets.
# Im gonna build a regression model to predict medv or median house value using 13 predictors such as,
# rmvar = average number of rooms per house
# age = proportion of owner-occupied units built prior to 1940
# lstat = percent of household with low socioeconomic status

Boston = load_data("Boston")
Boston.columns

Index(['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax',
       'ptratio', 'lstat', 'medv'],
      dtype='str')

In [11]:
# sm.OLS() = to fit a simple linear regression model
# medv and Lstat will be the single predictor.

X = pd.DataFrame({'intercept': np.ones(Boston.shape[0]),
                  'lstat': Boston['lstat']})
X[:4]

,intercept,lstat
0,1.0,4.98
1,1.0,9.14
2,1.0,4.03
3,1.0,2.94


In [12]:
# Extract the response and fit the model

y = Boston['medv']
model = sm.OLS(y, X)
results = model.fit()

In [ ]:
# sm.OLS() specifies the model and then model.fit() does the actual fitting
# summarize = parameter estimate, their standard error, t-statistics, p-values

summarize(results)

,coef,std err,t,P>|t|
intercept,34.5538,0.563,61.415,0.0
lstat,-0.9500,0.039,-24.528,0.0


In [15]:
""" Using Transformation: Fit and Transform """

# We often fit models with more than one predictor, typically selected from an array of data frame.
# Introduce transformations to the varaibles before fitting the model, specify interactions between variables and expand some particular variables into sets of variables = polynomials
# sklearn have this notion.
# A transform is an object that is created woth some parameters as arguments. = fit() + transform()
# ModelSpec() renamed MS() in the preamble creates a transform object,
# and then a pair of methods transform() and fit() are used to construct a correspounding model matrix.

' Using Transformation: Fit and Transform '

In [16]:
# The transform is created by design = MS(['lstat])
# the fit() method takes the original array and may do some initial computations on it
# the transform() method applies the fitted transformation to the array of data, and produces the model matrix.

design = MS(['lstat'])
design = design.fit(Boston)
X = design.transform(Boston)
X[:4]

,intercept,lstat
0,1.0,4.98
1,1.0,9.14
2,1.0,4.03
3,1.0,2.94


In [17]:
# These 2 operations can be combined with the fit_transform()

design = MS(['lstat'])
X = design.fit_transform(Boston)
X[:4]

,intercept,lstat
0,1.0,4.98
1,1.0,9.14
2,1.0,4.03
3,1.0,2.94


In [18]:
# If the two steps were done separately, the design object is changed as a result of the fit() operation.
# The power of this pipeline will become clearer when I fit more complex models that involve interaction and transformations.

results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   medv   R-squared:                       0.544
Model:                            OLS   Adj. R-squared:                  0.543
Method:                 Least Squares   F-statistic:                     601.6
Date:                Tue, 20 Oct 2026   Prob (F-statistic):           5.08e-88
Time:                        20:49:21   Log-Likelihood:                -1641.5
No. Observations:                 506   AIC:                             3287.
Df Residuals:                     504   BIC:                             3295.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
intercept     34.5538      0.563     61.415      0.000      33.448      35.659
lstat         -0.9500      0.039    -24.528      0.000      -1.026      -0.874
==============================================================================
Omnibus:                      137.043   Durbin-Watson:                   0.892
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              291.373
Skew:                           1.453   Prob(JB):                     5.36e-64
Kurtosis:                       5.319   Cond. No.                         29.7
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [19]:
# The fitted coefficients can also be retrieved as the params attribute of results

results.params

intercept    34.553841
lstat        -0.950049
dtype: float64

In [20]:
# get_prediction() can be used to obtain predictions and produce confidence intervals and prediction intervals for the prediction of medv for given values of lstat
# Try create a new data frame containing only the varaible lstat,
# with the values for this variable at which we wsih to make prediction.
# then use transform() method of design to create the corresponding model matrix

new_df = pd.DataFrame({'lstat':[5, 10, 15]})
newX = design.transform(new_df)
newX

,intercept,lstat
0,1.0,5
1,1.0,10
2,1.0,15


In [21]:
# Compute the prediction at newX and view them by extracting the predicted_mean attribute

new_predictions = results.get_prediction(newX)
new_predictions .predicted_mean

array([29.80359411, 25.05334734, 20.30310057])

In [22]:
# Prediction intervals are computing by setting obs=True

new_predictions.conf_int(obs=True, alpha=0.05)

array([[17.56567478, 42.04151344],
       [12.82762635, 37.27906833],
       [ 8.0777421 , 32.52845905]])

In [23]:
# The confidence and prediction intervals are centerd around the same point but the latter are substantially wider.